<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Lab's Diagrams

Every diagram in `docs/figures/` whose name starts `diagram_` is drawn here, as plain SVG written by Python with the
standard library alone, so that it needs no tool beyond the notebook's own pins (`notebooks/requirements.txt`) and
diffs as text. The drawing is deterministic: integer coordinates, a fixed element order, no timestamps or random
identifiers, so running this notebook again on an unchanged tree changes no byte of any figure. The last cell checks
that, by drawing every diagram twice and comparing the bytes.

Run with `jupyter nbconvert --to notebook --execute --inplace notebooks/figures.ipynb`.

In [1]:
import os
from html import escape

# nbconvert runs a notebook from its own directory; the figures live under the repository's root.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
FIGURES = "docs/figures"
FONT = "DejaVu Sans, Verdana, Arial, sans-serif"

# One palette for every diagram: a repository or a role keeps its colour from figure to figure.
PALETTE = {
    "world": ("#e8f1fb", "#2f6db3"),   # Master Control, the world server
    "wire": ("#eef7ee", "#3d8b40"),    # Link, the wire
    "grid": ("#fdf1e3", "#c46f12"),    # tron-grid-lite, the Grid
    "brain": ("#f6ecfa", "#8a3fa8"),   # a Program, a creature's brain
    "lab": ("#fff8d9", "#9a7b00"),     # iso-lab's own tools and documents
    "data": ("#eef0f2", "#4d5a66"),    # data from outside: the real worms
    "note": ("#ffffff", "#7a7a7a"),    # a plain note
}


def species(line):
    """Escape a line of text, setting the worm's binomial name in italics, as a species name is written."""
    name = "C. elegans"
    return f'<tspan font-style="italic">{name}</tspan>'.join(escape(part) for part in line.split(name))


class Svg:
    """A tiny SVG writer: rounded boxes, arrows and text, every number an integer."""

    def __init__(self, width, height, title):
        self.width, self.height, self.title = width, height, title
        self.parts = []

    def box(self, x, y, w, h, title, lines=(), role="note", dashed=False):
        fill, stroke = PALETTE[role]
        dash = ' stroke-dasharray="6 4"' if dashed else ""
        self.parts.append(
            f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="10" fill="{fill}" stroke="{stroke}" stroke-width="2"{dash}/>'
        )
        cx = x + w // 2
        ty = y + 26 if lines else y + h // 2 + 6
        self.parts.append(
            f'<text x="{cx}" y="{ty}" text-anchor="middle" font-size="16" font-weight="bold" fill="#1d1d1d">{escape(title)}</text>'
        )
        for i, line in enumerate(lines):
            self.parts.append(
                f'<text x="{cx}" y="{ty + 22 + 19 * i}" text-anchor="middle" font-size="13" fill="#333333">{species(line)}</text>'
            )

    def arrow(self, points, label=None, label_at=None, dashed=False, both=False, colour="#444444"):
        d = "M " + " L ".join(f"{x} {y}" for x, y in points)
        dash = ' stroke-dasharray="6 4"' if dashed else ""
        start = ' marker-start="url(#head)"' if both else ""
        self.parts.append(
            f'<path d="{d}" fill="none" stroke="{colour}" stroke-width="2"{dash}{start} marker-end="url(#head)"/>'
        )
        if label:
            lx, ly = label_at
            self.parts.append(
                f'<text x="{lx}" y="{ly}" text-anchor="middle" font-size="13" font-style="italic" fill="#333333">{escape(label)}</text>'
            )

    def text(self, x, y, s, size=13, anchor="middle", bold=False, colour="#333333"):
        weight = ' font-weight="bold"' if bold else ""
        self.parts.append(
            f'<text x="{x}" y="{y}" text-anchor="{anchor}" font-size="{size}"{weight} fill="{colour}">{escape(s)}</text>'
        )

    def render(self):
        head = (
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{self.width}" height="{self.height}" '
            f'viewBox="0 0 {self.width} {self.height}" font-family="{FONT}">\n'
            f"<title>{escape(self.title)}</title>\n"
            '<defs><marker id="head" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" '
            'orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#444444"/></marker></defs>\n'
            f'<rect x="0" y="0" width="{self.width}" height="{self.height}" fill="#ffffff"/>\n'
        )
        return head + "\n".join(self.parts) + "\n</svg>\n"


DIAGRAMS = {}


def diagram(name):
    def register(draw):
        DIAGRAMS[name] = draw
        return draw
    return register

## How the Pieces Fit

The organisation's repositories and the lab's own tools, and what flows between them. A creature's brain - a Program -
sees the world only through the senses the Grid renders for its body, and acts only through that body; the world
itself is Master Control's, reached over Link. What a life leaves behind - the Disk and the log - and the recordings
of real worms meet in the lab's tools, where the benchmark compares them. A dashed box or arrow is planned, not built.

In [2]:
@diagram("diagram_pieces.svg")
def pieces():
    s = Svg(1100, 620, "How the pieces fit")
    s.text(550, 34, "How the pieces fit", size=22, bold=True, colour="#1d1d1d")
    s.text(550, 58, "a creature lives on the Grid; the lab compares its life with real worms")

    # The world, left to right: Master Control - Link - the Grid - a Program.
    s.box(30, 90, 220, 120, "Master Control", ["the world server: physics,", "food, the tick, the logs"], role="world")
    s.box(320, 110, 150, 80, "Link", ["the wire"], role="wire")
    s.box(540, 90, 230, 120, "The Grid", ["renders each body's senses", "and applies its actions"], role="grid")
    s.box(860, 90, 210, 120, "A Program", ["a creature's brain,", "e.g. the placeholder worm"], role="brain")
    s.arrow([(250, 150), (318, 150)], both=True)
    s.arrow([(470, 150), (538, 150)], both=True)
    s.arrow([(770, 125), (858, 125)], "senses", (814, 115))
    s.arrow([(858, 175), (772, 175)], "actions", (814, 198))

    # What a life leaves behind.
    s.box(30, 300, 220, 100, "Disk and log", ["a life, recorded"], role="world")
    s.arrow([(140, 210), (140, 298)], "writes", (172, 260))
    s.box(30, 480, 220, 100, "Clu", ["replays a log and checks", "it gives the same world"], role="world")
    s.arrow([(140, 400), (140, 478)], "replay", (170, 444))

    # The real worms and the lab's tools.
    s.box(860, 300, 210, 100, "Real worms", ["recordings of C. elegans", "(Zenodo, public)"], role="data")
    s.box(540, 300, 230, 100, "fetch, features", ["the lab's Rust tools:", "download, then measure"], role="lab")
    s.arrow([(858, 350), (772, 350)], "download", (814, 340))
    s.arrow([(250, 350), (538, 350)], "read the Disk", (394, 340), dashed=True)
    s.text(394, 372, "(Disk reader planned)", size=12, colour="#7a7a7a")
    s.box(540, 480, 230, 100, "The benchmark", ["is the Grid worm's behaviour", "within the real worms'?"], role="lab", dashed=True)
    s.arrow([(655, 400), (655, 478)], "both sets of measures", (740, 444), dashed=True)
    s.text(655, 600, "(planned: not yet built or frozen)", size=12, colour="#7a7a7a")
    return s

## A Night's Cycle

How each task of a night lands, from `.claude/CLAUDE.md` § *Each run, in order*. A task is landed or honestly parked:
after three rounds of fixes that still end with CI red or *changes needed*, the pull request stays open and the task
goes under STATE.md's *Blocked*, with the next idea to try. Every box here is a rule already in force, so none is
dashed; in the other diagrams a dashed box is planned, not built.

In [3]:
@diagram("diagram_night.svg")
def night():
    s = Svg(1100, 720, "A night's cycle")
    s.text(550, 34, "A night's cycle", size=22, bold=True, colour="#1d1d1d")
    s.text(550, 58, "each task lands through the same steps, or is honestly parked")
    W, H = 230, 70
    # The main line, left to right then back: read, pick, branch, work, check, write down, open the pull request.
    s.box(30, 90, W, H, "1. Start from main", ["fetch, check out, pull"], role="lab")
    s.box(300, 90, W, H, "2. Read", ["STATE.md, the plan, the journal"], role="lab")
    s.box(570, 90, W, H, "3. Pick one task", ["about 45 minutes; split if bigger"], role="lab")
    s.box(840, 90, W, H, "4. Branch", ["claude/<short-name>"], role="lab")
    s.arrow([(260, 125), (298, 125)])
    s.arrow([(530, 125), (568, 125)])
    s.arrow([(800, 125), (838, 125)])
    s.arrow([(955, 160), (955, 208)])
    s.box(840, 210, W, H, "5. Do the work", ["pre-register; record every run"], role="lab")
    s.box(570, 210, W, H, "6-7. Check", ["review as a sceptic; lint, tests"], role="lab")
    s.box(300, 210, W, H, "8. Write it down", ["STATE.md and the journal"], role="lab")
    s.box(30, 210, W, H, "9. Pull request", ["commit, push, open it"], role="lab")
    s.arrow([(838, 245), (802, 245)])
    s.arrow([(568, 245), (532, 245)])
    s.arrow([(298, 245), (262, 245)])
    s.text(415, 178, "on a night's first task, also the new ideas and the audits", size=12, colour="#7a7a7a")
    # The gate: CI and a second agent; a failure of either goes to the fix loop.
    s.box(270, 340, W, H, "10-11. CI, then a review", ["CI Success; a fresh verifier"], role="lab")
    s.arrow([(145, 280), (145, 375), (268, 375)])
    s.box(680, 340, W, H, "Fix what is wrong", ["or answer why not"], role="note")
    s.arrow([(500, 365), (678, 365)])
    s.text(590, 352, "CI red, or changes needed", size=13, colour="#333333")
    s.arrow([(760, 410), (760, 440), (440, 440), (440, 412)])
    s.text(600, 458, "push; after a substantial fix it is looked at again", size=12, colour="#333333")
    s.box(270, 490, W, H, "12. Merge", ["the branch is deleted"], role="lab")
    s.arrow([(330, 410), (330, 488)])
    s.text(320, 470, "green, verified", size=13, anchor="end", colour="#333333")
    s.box(680, 490, 250, 88, "13. Park it", ["after three rounds: stays open,", "recorded under Blocked"], role="note")
    s.arrow([(870, 410), (870, 488)])
    s.text(882, 452, "after 3 rounds", size=12, anchor="start", colour="#7a7a7a")
    s.box(270, 620, W, 80, "14. Next task, or stop", ["at the hour, or when nothing is", "left, Blocked retried first"], role="lab")
    s.arrow([(385, 560), (385, 618)])
    s.arrow([(805, 578), (805, 660), (502, 660)])
    s.arrow([(268, 660), (14, 660), (14, 125), (28, 125)])
    s.text(140, 650, "the next task", size=13, colour="#333333")
    return s

## The Benchmark's Pipeline

How a verdict on the Grid's worm will be reached, from [docs/PLAN.md](../docs/PLAN.md) (*What "Indistinguishable"
Will Mean*, and Phase 0) and [docs/benchmark/FEATURES.md](../docs/benchmark/FEATURES.md). The real worms' recordings
and the Grid worm's lives pass through the same measures; the test, a classifier two-sample test frozen before any
brain is scored, asks whether a classifier can tell them apart on all the measures at once, and per-measure
comparisons say where they differ. A dashed box or arrow is planned, not yet built.

In [4]:
@diagram("diagram_benchmark.svg")
def benchmark():
    s = Svg(1190, 560, "The benchmark's pipeline")
    s.text(595, 34, "The benchmark's pipeline", size=22, bold=True, colour="#1d1d1d")
    s.text(595, 58, "real worms and the Grid's worm, measured the same way, then told apart if they can be")
    W, H = 250, 90
    # Real worms, top row.
    s.box(30, 90, W, H, "Real worms", ["recordings of wild-type", "C. elegans (public)"], role="data")
    s.box(320, 90, W, H, "fetch-block", ["download a block;", "keep files whose MD5 matches"], role="lab")
    s.box(610, 90, W, H, "features", ["speed and states, checked;", "durations and rates, not yet"], role="lab")
    s.box(900, 90, W, H, "Real worms' measures", ["the reference set", "(STATE.md item 2 (d))"], role="lab", dashed=True)
    s.arrow([(280, 135), (318, 135)])
    s.arrow([(570, 135), (608, 135)])
    s.arrow([(860, 135), (898, 135)], dashed=True)
    # The Grid's worm, bottom row.
    s.box(30, 280, W, H, "A Program's lives", ["on the Grid; so far one", "placeholder life"], role="brain")
    s.box(320, 280, W, H, "Disk", ["each life, recorded", "by Master Control"], role="world")
    s.box(610, 280, W, H, "features", ["the same measures,", "via a Disk reader"], role="lab", dashed=True)
    s.box(900, 280, W, H, "Grid worm's measures", ["the same measures", "over many lives"], role="lab", dashed=True)
    s.arrow([(280, 325), (318, 325)])
    s.arrow([(570, 325), (608, 325)], dashed=True)
    s.arrow([(860, 325), (898, 325)], dashed=True)
    # The test, frozen before any brain is scored.
    s.box(900, 430, W, 100, "The test", ["can a classifier tell them apart,", "on all measures at once?"], role="lab", dashed=True)
    s.arrow([(1150, 135), (1170, 135), (1170, 480), (1152, 480)], dashed=True)
    s.arrow([(1025, 370), (1025, 428)], dashed=True)
    s.text(870, 462, "frozen before any brain is scored;", size=13, anchor="end", colour="#333333")
    s.text(870, 482, "per-measure comparisons then", size=13, anchor="end", colour="#333333")
    s.text(870, 502, "say where the two differ", size=13, anchor="end", colour="#333333")
    return s

## From an Idea to a Brain

How an idea travels through the lab, from [docs/hypotheses/matej-hunches.md](../docs/hypotheses/matej-hunches.md)
(*Promote, do not copy*), [docs/hypotheses/experiments/README.md](../docs/hypotheses/experiments/README.md), STATE.md
item 6 and `.claude/CLAUDE.md` (*Ideas from people*, *Pre-register and ablate*, *Lean brains*). Nothing that fails
is dropped: it is kept, with the next idea to try.

In [5]:
@diagram("diagram_ideas.svg")
def ideas():
    s = Svg(1110, 520, "From an idea to a brain")
    s.text(555, 34, "From an idea to a brain", size=22, bold=True, colour="#1d1d1d")
    s.text(555, 58, "every idea is tested before it is trusted; what fails is kept, not dropped")
    W = 230
    # Where ideas come from: outside the lab's own work, so the colour of outside input.
    s.box(30, 90, W, 90, "Matej's hunches", ["128 ideas, A to G", "(matej-hunches.md)"], role="data")
    s.box(30, 230, W, 90, "Ideas from people", ["issues labelled idea,", "answered every night"], role="data")
    # The lab's own work: a plan, a test, a hypothesis.
    s.box(320, 150, W, 110, "A plan", ["written before any run:", "question, method, controls,", "what would refute it"], role="lab")
    s.arrow([(260, 135), (318, 180)])
    s.arrow([(260, 275), (318, 230)])
    s.box(610, 150, W, 110, "A test", ["on the toy worm, in a notebook,", "or on the Grid; results", "in the journal"], role="lab")
    s.arrow([(550, 205), (608, 205)])
    s.box(890, 90, 200, 110, "A hypothesis", ["promoted, not copied:", "H1, H2, ... or", "a task in STATE.md"], role="lab")
    s.arrow([(840, 180), (888, 150)])
    s.text(872, 224, "survives", size=13, colour="#333333")
    # Into a creature's brain, a Program: the colour of a Program.
    s.box(890, 330, 200, 110, "In a brain", ["a module stays only if", "switching it off costs", "the worm on the benchmark"], role="brain")
    s.arrow([(990, 200), (990, 328)], "earns its place", (1048, 270))
    s.box(610, 360, W, 100, "Kept, not dropped", ["To Revisit or Blocked,", "with the next idea to try"], role="note")
    s.arrow([(725, 260), (725, 358)], "fails or unclear", (790, 314))
    s.arrow([(608, 410), (435, 410), (435, 262)])
    s.text(522, 402, "tried again", size=12, colour="#7a7a7a")
    return s

## The Toy Worm's Graded-Gain Series

Every contrast [README.md § H14](../docs/hypotheses/README.md#h14-graded-slow-knobs-change-detecting-senses-and-individuality)
records for the toy worm's graded hunger gain, in two panels whose scope is stated on the figure. Each estimate is a
difference in late energy (hours 24 to 48) between two arms, with its 95% interval, read from the journal of
2026-10-30 at the section named in the code beside it. A contrast whose rule *holds* is blue, one that *fails*
orange, a descriptive one grey. In the rich layouts, K4 and W1 are judged in each layout against a margin of -0.03,
drawn dashed; every other rule asks for an interval wholly above zero. Half of K3's worms are K1's, so K3 is not an
independent replication; the new body seed's worms alone are the descriptive row beneath it.

In [6]:
import matplotlib

matplotlib.use("svg")
import matplotlib.pyplot as plt

plt.rcParams["svg.hashsalt"] = "iso-lab"
plt.rcParams["font.family"] = "DejaVu Sans"

# label, contrast, estimate, interval low and high, verdict, and the journal of 2026-10-30's section it is read from.
BANDED = [
    ("K1", "graded - greedy gated", 0.0377, 0.0046, 0.0678, "holds", "H14's Graded Hunger Gain on Ten Fresh Layouts"),
    ("K2", "graded - never steering", -0.0279, -0.0723, 0.0151, "descriptive", "H14's Graded Hunger Gain on Ten Fresh Layouts"),
    ("K3", "graded - greedy gated, 2 body seeds", 0.0441, 0.0149, 0.0737, "holds",
     "The Graded Hunger Gain: a Second Body Seed, and the Rich Layouts"),
    ("K3 new", "graded - greedy gated, new body seed only", 0.0506, 0.0161, 0.0892, "descriptive",
     "The Graded Hunger Gain: a Second Body Seed, and the Rich Layouts"),
    ("K3 run", "graded - never steering, 2 body seeds", -0.0305, -0.0725, 0.0111, "descriptive",
     "The Graded Hunger Gain: a Second Body Seed, and the Rich Layouts"),
    ("L1", "graded - un-latched", 0.0440, 0.0096, 0.0768, "holds", "Graded or Un-latched? An Ablation of the Graded Hunger Gain"),
    ("L2", "un-latched - latched gate", -0.0063, -0.0275, 0.0140, "fails", "Graded or Un-latched? An Ablation of the Graded Hunger Gain"),
    ("M1", "graded - scaled step", 0.0018, -0.0298, 0.0362, "fails", "Grading or Smaller Turns? A Scaled On/Off Gain"),
    ("M2", "scaled step - un-latched", 0.0423, 0.0116, 0.0725, "holds", "Grading or Smaller Turns? A Scaled On/Off Gain"),
    ("P1", "graded - matched step", -0.0056, -0.0365, 0.0218, "fails", "Grading or Smaller Turns, With the Constant Matched"),
    ("P2", "matched step - un-latched", 0.0496, 0.0198, 0.0772, "holds", "Grading or Smaller Turns, With the Constant Matched"),
    ("Q1", "graded - matched step, 3 body seeds", 0.0015, -0.0202, 0.0216, "fails",
     "Graded, Matched and Un-latched With Three Body Seeds per Layout"),
    ("Q2", "matched step - un-latched, 3 body seeds", 0.0471, 0.0281, 0.0644, "holds",
     "Graded, Matched and Un-latched With Three Body Seeds per Layout"),
]
K4 = "The Graded Hunger Gain: a Second Body Seed, and the Rich Layouts"
W1 = "The Matched Scaled Step in the Rich Layouts"
RICH = [
    ("K4 82200", "graded - never steering", 0.0026, -0.0010, 0.0061, "holds", K4),
    ("K4 82201", "graded - never steering", 0.0056, 0.0023, 0.0089, "holds", K4),
    ("K4 82202", "graded - never steering", 0.0002, -0.0035, 0.0035, "holds", K4),
    ("K4 82203", "graded - never steering", -0.0028, -0.0094, 0.0030, "holds", K4),
    ("K4 82204", "graded - never steering", 0.0055, 0.0048, 0.0062, "holds", K4),
    ("W1 82200", "matched step - never steering", -0.0017, -0.0044, 0.0010, "holds", W1),
    ("W1 82201", "matched step - never steering", -0.0016, -0.0043, 0.0009, "holds", W1),
    ("W1 82202", "matched step - never steering", 0.0001, -0.0032, 0.0035, "holds", W1),
    ("W1 82203", "matched step - never steering", -0.0001, -0.0023, 0.0019, "holds", W1),
    ("W1 82204", "matched step - never steering", -0.0014, -0.0046, 0.0005, "holds", W1),
    ("Z1", "graded - zeroed below 0.3", 0.0025, -0.0012, 0.0053, "fails", "The Graded Gain Without Its Steering Below 0.3"),
    ("F1", "floor - matched step", 0.0037, -0.0020, 0.0076, "fails",
     "A Constant Floor Below 0.3, With Three Body Seeds in the Rich Layouts"),
    ("F2", "graded - floor", -0.0022, -0.0052, 0.0005, "fails",
     "A Constant Floor Below 0.3, With Three Body Seeds in the Rich Layouts"),
    ("H1", "floor - matched step, 10 fresh rich layouts", 0.0021, -0.0014, 0.0052, "fails",
     "The Constant Floor on Ten Fresh Rich Layouts"),
]
COLOUR = {"holds": "#2f6db3", "fails": "#c46f12", "descriptive": "#7a7a7a"}


def panel(ax, rows, title, margin_rows=None):
    n = len(rows)
    for i, (_, _, est, lo, hi, verdict, _) in enumerate(rows):
        y = n - 1 - i
        ax.plot([lo, hi], [y, y], color=COLOUR[verdict], linewidth=2.2, solid_capstyle="butt")
        ax.plot([est], [y], marker="o", color=COLOUR[verdict], markersize=6)
    ax.axvline(0, color="#1d1d1d", linewidth=1)
    if margin_rows is not None:
        first, last = margin_rows
        top, bottom = n - 1 - first + 0.4, n - 1 - last - 0.4
        ax.plot([-0.03, -0.03], [bottom, top], color="#1d1d1d", linewidth=1, linestyle="--")
        ax.text(-0.0295, bottom - 0.35, "K4 and W1's margin, -0.03", fontsize=8, color="#1d1d1d", va="top")
    ax.set_yticks(range(n))
    ax.set_yticklabels([f"{r[0]}: {r[1]}" for r in reversed(rows)], fontsize=8.5)
    ax.set_title(title, fontsize=10.5, loc="left")
    ax.set_xlabel("difference in late energy (hours 24 to 48), with its 95% interval", fontsize=9)
    ax.tick_params(axis="x", labelsize=8.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def toyworm_plot():
    fig, (a, b) = plt.subplots(2, 1, figsize=(9, 9.6), gridspec_kw={"height_ratios": [len(BANDED), len(RICH) + 1]})
    panel(a, BANDED, "Ten fresh banded layouts, cold and warm pooled:\nthe leads beside the results against never steering")
    panel(b, RICH, "Rich layouts, on a finer scale:\nK4 and W1 per layout, then the pooled contrasts", margin_rows=(0, 9))
    handles = [plt.Line2D([], [], color=c, marker="o", linewidth=2.2, label=v) for v, c in COLOUR.items()]
    fig.legend(handles=handles, loc="upper right", fontsize=8.5, frameon=False, ncol=3)
    fig.suptitle("The toy worm's graded hunger gain: every contrast H14 records", fontsize=12, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    return fig


def save_plot(name):
    path = os.path.join(FIGURES, name)
    fig = toyworm_plot()
    fig.savefig(path, format="svg", metadata={"Date": None}, facecolor="#ffffff")
    plt.close(fig)
    return path


PLOTS = {"plot_toyworm_graded.svg": save_plot}

## Drawing, and the Check That Nothing Moves

Every diagram is drawn twice and the two drawings compared, which catches a drawing that varies within one run. Before
the new drawing is written, it is compared with the file already in `docs/figures/`, and the cell reports each figure
as *unchanged*, *changed* or *new*. A figure reported *changed* when its code did not change means the drawing depends
on something besides this notebook's code, which only a separate run can show: the check that matters is the
notebook run again from a fresh kernel, then `git diff --exit-code docs/figures`, which must report nothing.

In [7]:
import hashlib

for name, draw in sorted(DIAGRAMS.items()):
    first, second = draw().render(), draw().render()
    assert first == second, f"{name} differs between two drawings in one run"
    path = os.path.join(FIGURES, name)
    before = open(path, encoding="utf-8").read() if os.path.exists(path) else None
    status = "new" if before is None else ("unchanged" if before == first else "changed")
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(first)
    digest = hashlib.sha256(first.encode("utf-8")).hexdigest()
    print(f"{path}: {status}, {len(first.encode('utf-8'))} bytes, sha256 {digest}")

for name, make in sorted(globals().get("PLOTS", {}).items()):
    path = os.path.join(FIGURES, name)
    before = open(path, encoding="utf-8").read() if os.path.exists(path) else None
    make(name)
    after = open(path, encoding="utf-8").read()
    make(name)
    assert open(path, encoding="utf-8").read() == after, f"{name} differs between two drawings in one run"
    status = "new" if before is None else ("unchanged" if before == after else "changed")
    digest = hashlib.sha256(after.encode("utf-8")).hexdigest()
    print(f"{path}: {status}, {len(after.encode('utf-8'))} bytes, sha256 {digest}")

docs/figures/diagram_benchmark.svg: unchanged, 5953 bytes, sha256 dccdfa64bd337b559820bf5c997b5fd7a2bc8ad4c4f29f6811d8b1a87a010123
docs/figures/diagram_ideas.svg: unchanged, 5219 bytes, sha256 a03643ef001c1f03a72ab344605f3fbcb8c95de1e354cba45431e996dd906997
docs/figures/diagram_night.svg: unchanged, 7403 bytes, sha256 3e72906e85f2e94686a88e7c5086a684a496f09fd3ac206c91fabf237bb437ad
docs/figures/diagram_pieces.svg: unchanged, 6371 bytes, sha256 656b21dac3684de3c282be2ec77232f5cf7bad19ac6f06c31e56b83e56660d4c


docs/figures/plot_toyworm_graded.svg: unchanged, 163636 bytes, sha256 3cc89366382d00cde6c98a83bdb368376cd1052283ec2cce08a7bf1b0ac90af4
